# LeetCode 1015 — Smallest Integer Divisible by K bằng Python

Link đề: https://leetcode.com/problems/smallest-integer-divisible-by-k/ · Độ khó: **Medium** · Chủ đề: Hash Table, Math

Skeleton LeetCode:

```python
class Solution:
    def smallestRepunitDivByK(self, k: int) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất** (độ dài, hoặc `-1`). Số chỉ gồm chữ số `1` ("repunit") có thể rất dài (tới `10^5` chữ số) nên **không** được dựng bằng số nguyên lớn trong lời giải thật; ta chỉ theo dõi **số dư** `r = (r·10 + 1) mod k`. Điểm cốt lõi: chỉ có `k` số dư nên nếu có đáp án thì độ dài `≤ k` (nguyên lý chuồng bồ câu); và đáp án tồn tại ⇔ `gcd(k, 10) = 1`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm độ dài nhỏ nhất `L` sao cho số `11…1` (`L` chữ số `1`) chia hết cho `k`; không có thì `-1`.
2. Nắm công cụ: số học đồng dư (`mod`), cập nhật số dư `r ← (10·r + 1) mod k`, nguyên lý chuồng bồ câu, tập đã thăm, **bậc nhân** (multiplicative order) của `10` theo modulus `9k`.
3. Đi từ **dựng số nguyên lớn và thử chia** → **lặp số dư tối đa `k` bước** → **loại sớm khi `k` chẵn hoặc chia hết cho 5** → **tập số dư đã thăm (phát hiện chu trình)** → **công thức bậc của `10` modulo `9k`**.
4. Hiểu mấu chốt: `R_L = (10^L − 1)/9`; `k | R_L ⇔ 9k | 10^L − 1`. Số dư lặp `r_{L+1} = (10·r_L + 1) mod k` chỉ có `k` giá trị nên hoặc chạm `0` trong `k` bước hoặc rơi vào chu trình không chứa `0`.
5. Tránh lỗi kinh điển: dựng số lớn (`'1' * L`) khiến chậm/tràn, sai ưu tiên toán tử (`r * 10 + 1 % k`), lặp chỉ `k − 1` bước (bỏ sót đáp án `= k`), lệch độ dài `±1`, dùng số thực cho luỹ thừa của `10`, trả `0` cho `k = 1`.

## 2. Đề bài & ví dụ

Cho số nguyên dương `k`. Tìm **độ dài** của số nguyên dương **nhỏ nhất** `n` sao cho `n` chia hết cho `k` và `n` chỉ gồm chữ số `1`. Trả về độ dài của `n`; nếu không có `n` như vậy, trả về `-1`.

Lưu ý: `n` có thể không vừa số nguyên 64 bit.

**Ví dụ 1:** `k = 1` → `1` (`n = 1`).

**Ví dụ 2:** `k = 2` → `-1` (mọi `n` toàn chữ số `1` đều lẻ).

**Ví dụ 3:** `k = 3` → `3` (`n = 111 = 3 · 37`).

**Ràng buộc:** `1 <= k <= 10^5`.

In [1]:
import math
import random
import time
from typing import List

examples = [(1, 1), (2, -1), (3, 3), (7, 6), (11, 2), (17, 16), (5, -1)]
for k, want in examples:
    print(f"k = {k} -> {want}")

k = 1 -> 1
k = 2 -> -1
k = 3 -> 3
k = 7 -> 6
k = 11 -> 2
k = 17 -> 16
k = 5 -> -1


## 3. Kiến thức nền

### 3.1 Repunit và số dư

`R_1 = 1`, `R_{L+1} = 10·R_L + 1`. Với modulus `k` ta có `R_{L+1} mod k = (10·(R_L mod k) + 1) mod k`: chỉ cần giữ **số dư** (một số `< k`), không cần số lớn.

In [2]:
def repunit(L):
    return int("1" * L)


k = 7
r = 0
for L in range(1, 7):
    r = (r * 10 + 1) % k
    print(f"L={L}: R_L = {repunit(L)}, R_L mod {k} = {repunit(L) % k}, số dư theo công thức = {r}")
    assert repunit(L) % k == r

L=1: R_L = 1, R_L mod 7 = 1, số dư theo công thức = 1
L=2: R_L = 11, R_L mod 7 = 4, số dư theo công thức = 4
L=3: R_L = 111, R_L mod 7 = 6, số dư theo công thức = 6
L=4: R_L = 1111, R_L mod 7 = 5, số dư theo công thức = 5
L=5: R_L = 11111, R_L mod 7 = 2, số dư theo công thức = 2
L=6: R_L = 111111, R_L mod 7 = 0, số dư theo công thức = 0


### 3.2 Nguyên lý chuồng bồ câu

Dãy số dư `r_1, r_2, …` chỉ nhận `k` giá trị `{0, …, k−1}` và `r_{L+1}` được xác định hoàn toàn bởi `r_L`. Nếu trong `k` bước đầu không có `r_L = 0` thì có hai chỉ số `a < b ≤ k` với `r_a = r_b ≠ 0`; dãy rơi vào chu trình **không chứa `0`** và sẽ không bao giờ chạm `0`. Vậy: có đáp án thì đáp án `≤ k`.

### 3.3 Khi nào không có đáp án?

Mọi repunit tận cùng bằng `1`: lẻ và không chia hết cho `5`. Nên nếu `k` chẵn hoặc `5 | k` thì `k ∤ R_L` với mọi `L` ⇒ `-1`. (Điều ngược lại — `gcd(k, 10) = 1` ⇒ có đáp án — được chứng minh ở mục 10.)

In [3]:
print([k for k in range(1, 31) if k % 2 == 0 or k % 5 == 0][:10], "...  không có đáp án")
print([k for k in range(1, 31) if math.gcd(k, 10) == 1], "<- có đáp án")

[2, 4, 5, 6, 8, 10, 12, 14, 15, 16] ...  không có đáp án
[1, 3, 7, 9, 11, 13, 17, 19, 21, 23, 27, 29] <- có đáp án


## 4. Ý tưởng nền tảng / trực giác

`k = 7`: số dư theo `L = 1, 2, …`: `1, 4, 6, 5, 2, 0` — chạm `0` ở `L = 6` ⇒ `111111 = 7 · 15873`.

`k = 3`: `1, 2, 0` ⇒ `L = 3`.

`k = 2`: `1, 1, 1, …` — rơi vào chu trình `{1}` không chứa `0` ⇒ `-1`.

Vậy chỉ cần mô phỏng số dư và dừng khi (a) `r = 0` ⇒ trả `L`; (b) đã qua `k` bước (hoặc gặp lại số dư) ⇒ `-1`.

In [4]:
def remainder_trace(k, limit=20):
    r, out = 0, []
    for L in range(1, limit + 1):
        r = (r * 10 + 1) % k
        out.append(r)
        if r == 0:
            break
    return out


for k in (7, 3, 2, 11, 13):
    print(f"k = {k:2d}: {remainder_trace(k)}")

k =  7: [1, 4, 6, 5, 2, 0]
k =  3: [1, 2, 0]
k =  2: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
k = 11: [1, 0]
k = 13: [1, 11, 7, 6, 9, 0]


## 5. Brute force — dựng số lớn và thử chia

**Ý tưởng:** theo đúng định nghĩa: `n = 1, 11, 111, …` (số nguyên lớn của Python); với mỗi `L` từ `1` tới `k` kiểm tra `n % k == 0`. Chi phí `O(k · L)` do phép `%` trên số lớn — chỉ dùng cho `k` nhỏ; là **oracle** đúng định nghĩa (không dùng đồng dư gia tăng).

In [5]:
def smallest_bigint(k):
    n = 1
    for L in range(1, k + 1):                      # chuồng bồ câu: đáp án nếu có thì <= k
        if n % k == 0:
            return L
        n = n * 10 + 1
    return -1


for k, want in examples:
    assert smallest_bigint(k) == want
print([smallest_bigint(k) for k, _ in examples])

[1, -1, 3, 6, 2, 16, -1]


In [6]:
# Code nộp LeetCode cho Brute force (số lớn; chậm khi k lớn)
class SolutionBigInt:
    def smallestRepunitDivByK(self, k: int) -> int:
        n = 1
        for L in range(1, k + 1):
            if n % k == 0:
                return L
            n = n * 10 + 1
        return -1

**Độ phức tạp:** `O(k²)` (mỗi phép `%` trên số có tới `k` chữ số) — với `k = 10^5` không dùng được.

**Nút thắt:** ta mang theo cả số `R_L` khổng lồ trong khi chỉ cần `R_L mod k`.

## 6. Bước đột phá

Giữ **số dư** `r = R_L mod k` và cập nhật `r ← (10·r + 1) mod k`. Lặp tối đa `k` bước: nếu `r = 0` trả `L`; hết `k` bước mà không có thì trả `-1`. Thời gian `O(k)`, bộ nhớ `O(1)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** lặp số dư tối đa `k` bước (đáp án chính), **Cách 3** loại sớm khi `k` chẵn hoặc chia hết cho `5`, **Cách 4** tập số dư đã thăm, **Cách 5** bậc của `10` modulo `9k`.

### Cách 2 — Lặp số dư tối đa `k` bước

**Giải thích từng đoạn:**

1. `r = 0`.
2. Với `L = 1..k`: `r = (r * 10 + 1) % k`; nếu `r == 0` trả `L`.
3. Hết vòng: `-1`.

In [7]:
def smallest_remainder(k):
    r = 0
    for L in range(1, k + 1):
        r = (r * 10 + 1) % k                       # R_L mod k từ R_{L-1} mod k
        if r == 0:
            return L
    return -1


for k, want in examples:
    assert smallest_remainder(k) == want
print([smallest_remainder(k) for k, _ in examples])

[1, -1, 3, 6, 2, 16, -1]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionRemainder:
    def smallestRepunitDivByK(self, k: int) -> int:
        r = 0
        for L in range(1, k + 1):
            r = (r * 10 + 1) % k
            if r == 0:
                return L
        return -1

**Độ phức tạp:** `O(k)` thời gian, `O(1)` bộ nhớ (với `k = 10^5` là `10^5` bước).

### Cách 3 — Loại sớm khi `k` chẵn hoặc chia hết cho `5`

**Ý tưởng:** nếu `k % 2 == 0` hoặc `k % 5 == 0` thì trả `-1` ngay (mục 3.3); còn lại chạy Cách 2. Tiết kiệm `k` bước cho `60%` số `k`.

In [9]:
def smallest_early_exit(k):
    if k % 2 == 0 or k % 5 == 0:
        return -1                                  # repunit tận cùng 1: lẻ và không chia hết cho 5
    r = 0
    for L in range(1, k + 1):
        r = (r * 10 + 1) % k
        if r == 0:
            return L
    return -1                                      # (không bao giờ xảy ra khi gcd(k, 10) = 1: mục 10)


for k, want in examples:
    assert smallest_early_exit(k) == want
print([smallest_early_exit(k) for k, _ in examples])

[1, -1, 3, 6, 2, 16, -1]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionEarlyExit:
    def smallestRepunitDivByK(self, k: int) -> int:
        if k % 2 == 0 or k % 5 == 0:
            return -1
        r = 0
        for L in range(1, k + 1):
            r = (r * 10 + 1) % k
            if r == 0:
                return L
        return -1

**Độ phức tạp:** `O(1)` cho `k` chẵn/chia hết `5`; `O(k)` còn lại.

### Cách 4 — Tập số dư đã thăm (phát hiện chu trình)

**Ý tưởng:** không dùng cận `k` bước mà dùng `visited`: nếu gặp lại một số dư đã thăm (mà chưa gặp `0`) thì dãy đã vào chu trình không chứa `0` ⇒ `-1`.

In [11]:
def smallest_visited(k):
    seen = set()
    r, L = 0, 0
    while True:
        L += 1
        r = (r * 10 + 1) % k
        if r == 0:
            return L
        if r in seen:                              # chu trình không chứa 0
            return -1
        seen.add(r)


for k, want in examples:
    assert smallest_visited(k) == want
print([smallest_visited(k) for k, _ in examples])

[1, -1, 3, 6, 2, 16, -1]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionVisited:
    def smallestRepunitDivByK(self, k: int) -> int:
        seen = set()
        r, L = 0, 0
        while True:
            L += 1
            r = (r * 10 + 1) % k
            if r == 0:
                return L
            if r in seen:
                return -1
            seen.add(r)

**Độ phức tạp:** `O(k)` thời gian và `O(k)` bộ nhớ (tập số dư).

### Cách 5 — Bậc của `10` modulo `9k`

**Ý tưởng:** `R_L = (10^L − 1)/9`. `k | R_L ⇔ 9k | 10^L − 1 ⇔ 10^L ≡ 1 (mod 9k)`. Khi `gcd(k, 10) = 1` thì `gcd(10, 9k) = 1` và đáp án là **bậc nhân** `ord_{9k}(10)` (số mũ dương nhỏ nhất `L` với `10^L ≡ 1`). Bậc là một ước của `φ(9k)`: bắt đầu `ord = φ(9k)` rồi với mỗi ước nguyên tố `p` của `φ`, chừng nào `ord % p == 0` và `10^{ord/p} ≡ 1` thì chia `ord` cho `p`. Thời gian `O(√k)` — không cần lặp `k` bước.

In [13]:
def factorize(n):
    f, p = {}, 2
    while p * p <= n:
        while n % p == 0:
            f[p] = f.get(p, 0) + 1
            n //= p
        p += 1
    if n > 1:
        f[n] = f.get(n, 0) + 1
    return f


def euler_phi(n):
    phi = n
    for p in factorize(n):
        phi = phi // p * (p - 1)
    return phi


def smallest_order(k):
    if math.gcd(k, 10) != 1:
        return -1
    m = 9 * k
    order = euler_phi(m)
    for p in factorize(order):
        while order % p == 0 and pow(10, order // p, m) == 1:
            order //= p
    return order


for k, want in examples:
    assert smallest_order(k) == want
print([smallest_order(k) for k, _ in examples])

[1, -1, 3, 6, 2, 16, -1]


In [14]:
# Code nộp LeetCode cho Cách 5
import math


class SolutionOrder:
    def smallestRepunitDivByK(self, k: int) -> int:
        if math.gcd(k, 10) != 1:
            return -1

        def factorize(n):
            f, p = {}, 2
            while p * p <= n:
                while n % p == 0:
                    f[p] = f.get(p, 0) + 1
                    n //= p
                p += 1
            if n > 1:
                f[n] = f.get(n, 0) + 1
            return f

        m = 9 * k
        phi = m
        for p in factorize(m):
            phi = phi // p * (p - 1)
        order = phi
        for p in factorize(phi):
            while order % p == 0 and pow(10, order // p, m) == 1:
                order //= p
        return order

**Độ phức tạp:** `O(√k + log² k)` thời gian (phân tích thừa số bằng chia thử `O(√(9k))`, các luỹ thừa `pow` `O(log)`), `O(1)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `k = 7`:

| `L` | `r` trước | `r = (10r + 1) mod 7` | `r == 0`? |
|---|---|---|---|
| 1 | 0 | `1` | không |
| 2 | 1 | `11 mod 7 = 4` | không |
| 3 | 4 | `41 mod 7 = 6` | không |
| 4 | 6 | `61 mod 7 = 5` | không |
| 5 | 5 | `51 mod 7 = 2` | không |
| 6 | 2 | `21 mod 7 = 0` | **có ⇒ trả 6** |

Hàm trace in từng bước:

In [15]:
def trace_remainder(k):
    print(f"k = {k}")
    r = 0
    for L in range(1, k + 1):
        prev = r
        r = (r * 10 + 1) % k
        print(f"  L={L}: ({prev}*10 + 1) mod {k} = {r}" + ("  => chia hết! trả " + str(L) if r == 0 else ""))
        if r == 0:
            return L
    print("  hết k bước mà chưa chạm 0 => -1")
    return -1


assert trace_remainder(7) == 6
assert trace_remainder(3) == 3

k = 7
  L=1: (0*10 + 1) mod 7 = 1
  L=2: (1*10 + 1) mod 7 = 4
  L=3: (4*10 + 1) mod 7 = 6
  L=4: (6*10 + 1) mod 7 = 5
  L=5: (5*10 + 1) mod 7 = 2
  L=6: (2*10 + 1) mod 7 = 0  => chia hết! trả 6
k = 3
  L=1: (0*10 + 1) mod 3 = 1
  L=2: (1*10 + 1) mod 3 = 2
  L=3: (2*10 + 1) mod 3 = 0  => chia hết! trả 3


**Trace trường hợp `-1`:** `k = 2`: `r = 1, 1, 1, …` — `k = 2` bước vẫn `1`, không chạm `0` ⇒ `-1` (số dư rơi vào chu trình `{1}`).

In [16]:
assert trace_remainder(2) == -1
print([smallest_visited(k) for k in (2, 4, 5, 6, 8, 10, 20)])

k = 2
  L=1: (0*10 + 1) mod 2 = 1
  L=2: (1*10 + 1) mod 2 = 1
  hết k bước mà chưa chạm 0 => -1
[-1, -1, -1, -1, -1, -1, -1]


## 9. Code nộp LeetCode

Chọn **Cách 3** (lặp số dư có loại sớm): đơn giản, `O(k)` và nhanh hơn Cách 2 cho `60%` số `k`. Giữ nguyên `class Solution`, `smallestRepunitDivByK(self, k)`:

In [17]:
class Solution:
    def smallestRepunitDivByK(self, k: int) -> int:
        if k % 2 == 0 or k % 5 == 0:                  # repunit tận cùng 1: lẻ và không chia hết cho 5
            return -1
        r = 0                                         # r = R_L mod k (không dựng số lớn)
        for L in range(1, k + 1):                     # chuồng bồ câu: nếu có đáp án thì L <= k
            r = (r * 10 + 1) % k
            if r == 0:
                return L
        return -1


print([Solution().smallestRepunitDivByK(k) for k, _ in examples])
assert all(Solution().smallestRepunitDivByK(k) == w for k, w in examples)

[1, -1, 3, 6, 2, 16, -1]


## 10. Vì sao đúng

**(a) Số dư gia tăng.** `R_{L+1} = 10·R_L + 1` nên `R_{L+1} mod k = (10·(R_L mod k) + 1) mod k`. Vậy `r_L` trong Cách 2 bằng `R_L mod k`, và `r_L = 0 ⇔ k | R_L`. Trả về `L` nhỏ nhất như vậy ⇒ độ dài nhỏ nhất.

**(b) Cận `L ≤ k`.** Xét `r_0 = 0, r_1, …, r_k` (`k + 1` giá trị trong `k` số dư). Nếu có đáp án thì `r_L = 0` với `L` nhỏ nhất. Giả sử `L > k`: khi đó `r_1, …, r_k` đều khác `0`, nên chỉ nhận `k − 1` giá trị ⇒ tồn tại `1 ≤ a < b ≤ k` với `r_a = r_b`. Vì `r_{i+1}` là hàm của `r_i`, dãy tuần hoàn từ `a` với chu kỳ `b − a` và không bao giờ chạm `0` (các giá trị lặp lại đều khác `0`) — mâu thuẫn với việc `r_L = 0` ở `L > k`. Vậy `L ≤ k`; lặp `k` bước đủ và `-1` ngay khi hết vòng là đúng.

**(c) `gcd(k, 10) ≠ 1 ⇒ -1`.** Nếu `2 | k` hoặc `5 | k` thì `R_L ≡ 1 (mod 2)` (chữ số cuối `1`) và `R_L ≡ 1 (mod 5)` (chữ số cuối `1`), nên `k ∤ R_L`. Do đó `-1`.

**(d) `gcd(k, 10) = 1 ⇒` có đáp án.** `R_L = (10^L − 1)/9`. `k | R_L ⇔ 9k | 10^L − 1` (vì `9 | 10^L − 1`). Nếu `gcd(k, 10) = 1` thì `gcd(10, 9k) = 1` (`10 = 2·5` và `9k` không chia hết cho `2`, `5`). Theo định lý Euler, `10^{φ(9k)} ≡ 1 (mod 9k)`; vậy `L = φ(9k)` thoả. Do đó có đáp án, và đáp án nhỏ nhất là **bậc** `ord_{9k}(10)`. Cũng suy ra Cách 3 không bao giờ rơi vào `return -1` cuối vòng khi `gcd(k, 10) = 1` (vì theo (b) đáp án `≤ k`).

**(e) Cách 4.** Nếu gặp lại số dư `r ≠ 0` đã thấy thì dãy vào chu trình không chứa `0` (theo (b)) ⇒ `-1`; nếu chạm `0` trả độ dài. Mỗi số dư tối đa một lần ⇒ kết thúc trong `≤ k` bước.

**(f) Cách 5.** Bậc `ord_m(10)` chia `φ(m)` (định lý Lagrange); bắt đầu từ `φ(m)` và loại các thừa số nguyên tố `p` chừng nào `10^{ord/p} ≡ 1` cho bậc đúng (thuật toán chuẩn). Kết quả bằng `L` nhỏ nhất với `10^L ≡ 1 (mod 9k)`, tức đáp án theo (d).

**(g) Tính chất kiểm chứng được.** (1) Có đáp án ⇔ `gcd(k, 10) = 1`; (2) đáp án `L` thoả `pow(10, L, 9k) == 1` và không có `L' < L` thoả; (3) nếu `k₁ | k₂` (cả hai nguyên tố cùng nhau với `10`) thì `ans(k₁) | ans(k₂)`; (4) nếu `gcd(k₁, k₂) = 1` thì `ans(k₁k₂) = lcm(ans(k₁), ans(k₂))`; (5) `ans(k) ≤ k`.

**(h) Kiểm chứng bằng máy.** Mọi cách khớp oracle số lớn (`smallest_bigint`) với mọi `k ≤ 600`; các cách số dư/ tập/ bậc khớp nhau với mọi `k ≤ 3000`; Cách 3 và Cách 5 khớp nhau trên mẫu `k` tới `10^5` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Số lớn | `O(k²)` | `O(k)` | oracle `k ≲ 600` |
| 2. Lặp số dư `k` bước | `O(k)` | `O(1)` | đơn giản |
| 3. Loại sớm + lặp | `O(k)` (`O(1)` nếu chẵn/5) | `O(1)` | **đáp án chính** |
| 4. Tập số dư | `O(k)` | `O(k)` | phát hiện chu trình |
| 5. Bậc của `10` mod `9k` | `O(√k)` | `O(1)` | số học, nhanh nhất |

Đo thực tế:

In [18]:
def bench(fn, k, repeat=3):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(k)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


for k in (99991, 99989, 77777 + 2, 100000):
    row = []
    for name, fn in [("remainder", smallest_remainder), ("early_exit", smallest_early_exit), ("visited", smallest_visited), ("order", smallest_order)]:
        row.append(f"{name} {bench(fn, k) * 1e3:7.2f} ms")
    print(f"k = {k:6d} -> {smallest_order(k):6d} | " + " | ".join(row))
print(f"bigint k = 600: {bench(smallest_bigint, 599, repeat=1) * 1e3:7.1f} ms")

k =  99991 ->  49995 | remainder    3.32 ms | early_exit    3.28 ms | visited    7.18 ms | order    0.02 ms


k =  99989 ->  99988 | remainder    6.71 ms | early_exit    6.65 ms | visited   13.56 ms | order    0.03 ms
k =  77779 ->    960 | remainder    0.06 ms | early_exit    0.06 ms | visited    0.12 ms | order    0.01 ms
k = 100000 ->     -1 | remainder    6.73 ms | early_exit    0.00 ms | visited    0.00 ms | order    0.00 ms
bigint k = 600:     0.1 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi `k ≤ 3000` (oracle số lớn cho `k ≤ 600`), (E) tính chất / ngẫu nhiên tới `10^5`, (F) giới hạn.

### A. Ví dụ của đề

In [19]:
IMPLS = {
    "remainder": smallest_remainder,
    "early_exit": smallest_early_exit,
    "visited": smallest_visited,
    "order": smallest_order,
    "bigint": smallest_bigint,
    "Solution": lambda k: Solution().smallestRepunitDivByK(k),
    "SolutionRemainder": lambda k: SolutionRemainder().smallestRepunitDivByK(k),
    "SolutionEarlyExit": lambda k: SolutionEarlyExit().smallestRepunitDivByK(k),
    "SolutionVisited": lambda k: SolutionVisited().smallestRepunitDivByK(k),
    "SolutionOrder": lambda k: SolutionOrder().smallestRepunitDivByK(k),
    "SolutionBigInt": lambda k: SolutionBigInt().smallestRepunitDivByK(k),
}
BIG = {"bigint", "SolutionBigInt"}


def check(k, want=None, names=None, big_limit=600):
    if want is None and k <= big_limit:
        want = smallest_bigint(k)
    results = {}
    for name in (names or IMPLS):
        if name in BIG and k > big_limit:
            continue
        results[name] = IMPLS[name](k)
    assert len(set(results.values())) == 1, (k, results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (k, got, want)
    return got


for k, want in examples:
    check(k, want)
print("ví dụ đề: oracle số lớn và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle số lớn và 11 cài đặt đúng


### B. Edge case

- `k = 1`: `1` (không phải `0`).
- `k = 2, 4, 5, 8, 10, 20, 25, 100000`: `-1`.
- Số nguyên tố nhỏ: `3 → 3`, `7 → 6`, `11 → 2`, `13 → 6`, `17 → 16`, `19 → 18`.
- `k` là luỹ thừa của `3`: `3, 9, 27` (đáp án `= k` ở `3`, `9`, `27`).
- `k = 9`: `9`; `k = 37`: `3`; `k = 41`: `5`; `k = 101`: `4`.
- `k = 99991` (nguyên tố lớn) và `k = 99999`, `k = 99997`.

In [20]:
edge = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 17, 19, 20, 25, 27, 37, 41, 101, 111, 333, 999, 1111]
for k in edge:
    check(k)
known = {1: 1, 3: 3, 7: 6, 9: 9, 11: 2, 13: 6, 17: 16, 19: 18, 27: 27, 37: 3, 41: 5, 101: 4, 111: 3, 239: 7}
for k, v in known.items():
    assert Solution().smallestRepunitDivByK(k) == v == smallest_order(k), (k, v)
for k in (100000, 99999, 99991, 99989, 99997):
    results = {name: IMPLS[name](k) for name in ("early_exit", "order", "Solution", "SolutionOrder", "SolutionEarlyExit")}
    assert len(set(results.values())) == 1, (k, results)
    print(f"  k = {k}: {results['Solution']}")
print(len(edge), "edge case nhỏ (oracle số lớn) + giá trị đã biết + k lớn: đúng")

  k = 100000: -1
  k = 99999: 45
  k = 99991: 49995
  k = 99989: 99988
  k = 99997: 7866
24 edge case nhỏ (oracle số lớn) + giá trị đã biết + k lớn: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [21]:
def bug_precedence(k):                                 # r * 10 + 1 % k (thiếu ngoặc): 1 % k tính trước
    r = 0
    for L in range(1, k + 1):
        r = r * 10 + 1 % k                          # BUG: 1 % k tính trước, r không được rút gọn
        if r == 0:
            return L
    return -1


def bug_range_k_minus_1(k):                            # lặp chỉ k - 1 bước: bỏ sót đáp án = k
    r = 0
    for L in range(1, k):
        r = (r * 10 + 1) % k
        if r == 0:
            return L
    return -1


def bug_length_off_by_one(k):                          # trả số bước 0-based (L - 1)
    r = 0
    for L in range(k):
        r = (r * 10 + 1) % k
        if r == 0:
            return L
    return -1


def bug_float_pow(k):                                  # dùng số thực cho 10**L
    for L in range(1, min(k, 40) + 1):
        n = int((10.0 ** L - 1) // 9)
        if n % k == 0:
            return L
    return -1


def bug_k1_zero(k):                                    # coi k = 1 là trường hợp "độ dài 0" (số rỗng)
    return 0 if k == 1 else smallest_remainder(k)


def bug_no_exit_infinite(k):                           # không có cận bước cho k chẵn: vòng lặp vô hạn (có giới hạn để minh hoạ)
    r, L = 0, 0
    while True:
        L += 1
        if L > 5000:
            raise RuntimeError("vòng lặp vô hạn")
        r = (r * 10 + 1) % k
        if r == 0:
            return L


def bug_only_even_excluded(k):                         # chỉ loại k chẵn, quên bội của 5, và dùng cận sai (while L < 10)
    if k % 2 == 0:
        return -1
    r = 0
    for L in range(1, 11):                             # cận cố định 10 bước: quá ít
        r = (r * 10 + 1) % k
        if r == 0:
            return L
    return -1


BUGS = [bug_precedence, bug_range_k_minus_1, bug_length_off_by_one, bug_float_pow, bug_k1_zero, bug_no_exit_infinite, bug_only_even_excluded]
for bug in BUGS:
    found = None
    for k in range(1, 200):
        try:
            got = bug(k)
        except RuntimeError:
            got = "RuntimeError"
        if got != smallest_bigint(k):
            found = (k, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:24s} sai trên k = {found[0]}: ra {found[1]}, đúng {smallest_bigint(found[0])}")

bug_precedence           sai trên k = 3: ra -1, đúng 3
bug_range_k_minus_1      sai trên k = 1: ra -1, đúng 1
bug_length_off_by_one    sai trên k = 1: ra 0, đúng 1
bug_float_pow            sai trên k = 18: ra 17, đúng -1
bug_k1_zero              sai trên k = 1: ra 0, đúng 1
bug_no_exit_infinite     sai trên k = 2: ra RuntimeError, đúng -1
bug_only_even_excluded   sai trên k = 17: ra -1, đúng 16


### D. Vét cạn mọi `k ≤ 3000`

Oracle số lớn (`smallest_bigint`) cho mọi `k ≤ 600`; với `600 < k ≤ 3000` các cách số dư / loại sớm / tập / bậc phải khớp nhau và khớp **tính chất độc lập**: `pow(10, L, 9k) == 1` và `L` nhỏ nhất (kiểm bằng cách thử mọi `L' < L`), `-1 ⇔ gcd(k, 10) > 1`. Với `k ≤ 600` kiểm cả `-1 ⇔ gcd > 1`.

In [22]:
t0 = time.perf_counter()
fast = [n for n in IMPLS if n not in BIG]
cnt = pos = 0
for k in range(1, 601):
    want = smallest_bigint(k)
    for name in IMPLS:
        assert IMPLS[name](k) == want, (name, k, IMPLS[name](k), want)
    assert (want == -1) == (math.gcd(k, 10) != 1)
    cnt += 1
    pos += want != -1
print(f"k ≤ 600: {cnt} giá trị, {len(IMPLS)} cài đặt khớp oracle số lớn; có đáp án: {pos};", round(time.perf_counter() - t0, 1), "s")

t0 = time.perf_counter()
for k in range(601, 3001):
    r = {name: IMPLS[name](k) for name in fast}
    assert len(set(r.values())) == 1, (k, r)
    L = r["Solution"]
    assert (L == -1) == (math.gcd(k, 10) != 1)
    if L != -1:
        assert pow(10, L, 9 * k) == 1 and L <= k                    # đúng bậc: 10^L ≡ 1 (mod 9k)
        assert all(pow(10, d, 9 * k) != 1 for d in range(1, L))      # nhỏ nhất
        assert repunit(L) % k == 0 if L <= 400 else True
print("601 ≤ k ≤ 3000: các cài đặt nhanh khớp nhau; điều kiện bậc & tính nhỏ nhất đúng;", round(time.perf_counter() - t0, 1), "s")

k ≤ 600: 600 giá trị, 11 cài đặt khớp oracle số lớn; có đáp án: 240; 0.2 s


601 ≤ k ≤ 3000: các cài đặt nhanh khớp nhau; điều kiện bậc & tính nhỏ nhất đúng; 1.1 s


### E. Tính chất / ngẫu nhiên tới `10^5` (seed cố định)

(1) `smallest_early_exit` khớp `smallest_order` trên mẫu `k` tới `10^5`; (2) `pow(10, L, 9k) == 1` và `L` là ước của `φ(9k)`; (3) **ước**: nếu `k₁ | k₂` (cùng nguyên tố cùng nhau với `10`) thì `ans(k₁) | ans(k₂)`; (4) **tích nguyên tố cùng nhau**: `ans(k₁k₂) = lcm(ans(k₁), ans(k₂))` khi `gcd(k₁, k₂) = 1`; (5) `ans(k) ≤ k`; (6) có đáp án ⇔ `gcd(k, 10) = 1`.

In [23]:
rng = random.Random(20240113)
t0 = time.perf_counter()
checked = 0
for _ in range(60):
    k = rng.randint(1, 100000)
    a, b = Solution().smallestRepunitDivByK(k), smallest_order(k)
    assert a == b, (k, a, b)                                                    # (1)
    assert (a != -1) == (math.gcd(k, 10) == 1)                                  # (6)
    if a != -1:
        assert pow(10, a, 9 * k) == 1 and euler_phi(9 * k) % a == 0 and a <= k  # (2) (5)
        checked += 1
for _ in range(300):
    k1, k2 = rng.randint(1, 3000), rng.randint(1, 3000)
    if math.gcd(k1, 10) != 1 or math.gcd(k2, 10) != 1:
        continue
    a1, a2 = smallest_order(k1), smallest_order(k2)
    if k2 % k1 == 0 or k1 % k2 == 0:                                             # (3)
        small, big = (k1, k2) if k2 % k1 == 0 else (k2, k1)
        assert smallest_order(big) % smallest_order(small) == 0
    if math.gcd(k1, k2) == 1:                                                    # (4)
        assert smallest_order(k1 * k2) == a1 * a2 // math.gcd(a1, a2)
for k in (3, 9, 27, 81, 243, 729):                                               # chuỗi ước: 3 | 9 | 27 | ...
    assert smallest_order(3 * k) % smallest_order(k) == 0
print("60 giá trị k ngẫu nhiên tới 10^5 (", checked, "có đáp án): khớp Cách 5; ước và tích nguyên tố cùng nhau đúng;", round(time.perf_counter() - t0, 1), "s")

60 giá trị k ngẫu nhiên tới 10^5 ( 18 có đáp án): khớp Cách 5; ước và tích nguyên tố cùng nhau đúng; 0.0 s


### F. Giới hạn

`k` tới `10^5`: các nguyên tố lớn (đáp án có thể rất gần `k − 1`), bội của `3` (`99999`), `k = 10^5` (`-1`); đo thời gian các cách số dư và so sánh với bậc; số `k` cho đáp án lớn nhất trong `[1, 10^5]` tìm bằng Cách 5.

In [24]:
t0 = time.perf_counter()
big_k = [99991, 99989, 99971, 99961, 99929, 99923, 99999, 99997, 100000, 65537, 30001, 77777]
for k in big_k:
    a = Solution().smallestRepunitDivByK(k)
    assert a == smallest_order(k) == (smallest_early_exit(k)), k
best_k, best_ans = 1, 1
for k in range(1, 100001):
    if k % 2 and k % 5:
        a = smallest_order(k)
        if a > best_ans:
            best_k, best_ans = k, a
print("k có đáp án lớn nhất trong [1, 10^5] (theo Cách 5):", best_k, "->", best_ans)
assert Solution().smallestRepunitDivByK(best_k) == best_ans and best_ans <= best_k
print("k lớn: các cách khớp; Cách 3 chạy được trong giới hạn thời gian;", round(time.perf_counter() - t0, 1), "s")

k có đáp án lớn nhất trong [1, 10^5] (theo Cách 5): 99989 -> 99988
k lớn: các cách khớp; Cách 3 chạy được trong giới hạn thời gian; 0.6 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: sai ưu tiên toán tử `r * 10 + 1 % k`

In [25]:
print("bug:", bug_precedence(3), bug_precedence(7), "| đúng:", Solution().smallestRepunitDivByK(3), Solution().smallestRepunitDivByK(7))
assert bug_precedence(7) != 6

bug: -1 -1 | đúng: 3 6


**Vì sao sai:** `%` có độ ưu tiên cao hơn `+`: `r * 10 + 1 % k = r * 10 + (1 % k)`, số dư không được rút gọn theo `k`. **Đúng:** `(r * 10 + 1) % k`.

### Lỗi 2: lặp chỉ `k − 1` bước

In [26]:
print("bug:", bug_range_k_minus_1(3), "| đúng:", Solution().smallestRepunitDivByK(3))
assert bug_range_k_minus_1(3) == -1 and Solution().smallestRepunitDivByK(3) == 3

bug: -1 | đúng: 3


**Vì sao sai:** đáp án có thể **bằng** `k` (ví dụ `k = 3`, `9`, `27`). Phải lặp đủ `k` bước (`range(1, k + 1)`).

### Lỗi 3: lệch độ dài (đếm 0-based)

In [27]:
print("bug:", bug_length_off_by_one(3), bug_length_off_by_one(7), "| đúng:", Solution().smallestRepunitDivByK(3), Solution().smallestRepunitDivByK(7))
assert bug_length_off_by_one(7) == 5 and Solution().smallestRepunitDivByK(7) == 6

bug: 2 5 | đúng: 3 6


**Vì sao sai:** sau `L` lần cập nhật số dư, số có **`L` chữ số**; đếm từ `0` làm lệch `1`.

### Lỗi 4: dùng số thực cho `10**L`

In [28]:
print("bug:", bug_float_pow(19), bug_float_pow(23), "| đúng:", Solution().smallestRepunitDivByK(19), Solution().smallestRepunitDivByK(23))
assert bug_float_pow(19) == -1 and Solution().smallestRepunitDivByK(19) == 18 and bug_float_pow(23) != Solution().smallestRepunitDivByK(23)

bug: -1 21 | đúng: 18 22


**Vì sao sai:** số thực `float` chỉ chính xác `~15–17` chữ số; repunit dài hơn bị làm tròn (ví dụ `k = 19` cần `18` chữ số). **Đúng:** chỉ dùng số **dư** nguyên.

### Lỗi 5: `k = 1` trả `0`

In [29]:
print("bug:", bug_k1_zero(1), "| đúng:", Solution().smallestRepunitDivByK(1))
assert bug_k1_zero(1) == 0 and Solution().smallestRepunitDivByK(1) == 1

bug: 0 | đúng: 1


**Vì sao sai:** số nhỏ nhất toàn chữ số `1` là `1` (độ dài `1`); độ dài `0` (số rỗng) không hợp lệ.

### Lỗi 6 (bonus): vòng lặp không có cận (k chẵn)

In [30]:
try:
    bug_no_exit_infinite(2)
    print("không lỗi?")
except RuntimeError as e:
    print("bug:", e)
print("đúng:", Solution().smallestRepunitDivByK(2), "| chỉ loại k chẵn, cận 10:", bug_only_even_excluded(7), bug_only_even_excluded(17), "| đúng:", Solution().smallestRepunitDivByK(17))
assert bug_only_even_excluded(17) == -1

bug: vòng lặp vô hạn
đúng: -1 | chỉ loại k chẵn, cận 10: 6 -1 | đúng: 16


**Vì sao sai:** với `k` không có đáp án, dãy số dư lặp mãi; cần cận `k` bước (hoặc loại trước `k` chẵn/chia hết `5` và dùng cận đủ lớn).

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "số có dạng đặc biệt (toàn chữ số `1`/`9`, ...) chia hết cho `k`" → mô phỏng **số dư**; số trạng thái hữu hạn (`k`) ⇒ chuồng bồ câu cho cận lặp.

**Mẹo:**

1. Không dựng số lớn: `r = (r * 10 + d) % k`.
2. Cận lặp `k` bước (đáp án có thể `= k`).
3. Loại sớm: `k` chẵn hoặc chia hết `5` ⇒ `-1` (với repunit).
4. Cách nhìn số học: `k | R_L ⇔ 10^L ≡ 1 (mod 9k)`.
5. Oracle: số nguyên lớn của Python cho `k` nhỏ.
6. Kiểm tra tính chất: ước, lcm.

**Họ bài cùng pattern:** Numbers With Repeated Digits (1012), Smallest Multiple With 0 and 1 (BFS số dư), Multiples of 3 and 5, Divisor Game, Largest Multiple of Three (1363), Remainder of the Number Formed by Digits.

## 15. Biến thể

1. **Chữ số `d` bất kỳ** (số `dd…d`).
2. **Bội nhỏ nhất của `k` chỉ gồm chữ số `0` và `1`** (BFS trên số dư).
3. **Đếm số `L ≤ N` thoả `R_L` chia hết cho `k`** (các bội của bậc).

Cài đặt và đối chiếu số lớn:

In [31]:
def smallest_digit_d(k, d):
    """Độ dài nhỏ nhất L sao cho số dd...d (L chữ số d) chia hết cho k; -1 nếu không có."""
    r = 0
    for L in range(1, k + 1):
        r = (r * 10 + d) % k
        if r == 0:
            return L
    return -1


def smallest_01_multiple(k):
    """Bội nhỏ nhất của k chỉ gồm chữ số 0, 1 (bắt đầu bằng 1): BFS trên số dư; trả chuỗi."""
    from collections import deque
    start = 1 % k
    parent = {start: (None, "1")}
    queue = deque([start])
    while queue:
        r = queue.popleft()
        if r == 0:
            s = []
            while r is not None:
                prev, ch = parent[r]
                s.append(ch)
                r = prev
            return "".join(reversed(s))
        for ch in "01":
            nr = (r * 10 + int(ch)) % k
            if nr not in parent:
                parent[nr] = (r, ch)
                queue.append(nr)
    return None


def count_L_upto(k, N):
    a = smallest_order(k)
    return 0 if a == -1 else N // a


def brute_digit_d(k, d):
    n = d
    for L in range(1, k + 1):
        if n % k == 0:
            return L
        n = n * 10 + d
    return -1


def brute_01_multiple(k, limit=14):
    for m in range(1, 2 ** limit):
        s = bin(m)[2:]                                  # chuỗi 0/1 bắt đầu bằng 1; tăng dần theo giá trị nhị phân = theo độ dài rồi từ điển
        if int(s) % k == 0:
            return s
    return None


for k in range(1, 120):
    for d in (1, 2, 3, 7, 9):
        assert smallest_digit_d(k, d) == brute_digit_d(k, d)                                   # (1)
    s = smallest_01_multiple(k)
    assert int(s) % k == 0 and set(s) <= {"0", "1"} and s[0] == "1"                           # (2) hợp lệ
    b = brute_01_multiple(k)
    if b is not None:
        assert s == b or (len(s) == len(b) and int(s) % k == 0)                               # (2) cùng độ dài ngắn nhất
for k in range(1, 600):
    N = 1000
    assert count_L_upto(k, N) == (N // smallest_order(k) if smallest_order(k) != -1 else 0)
for k in (3, 7, 9, 11, 13, 37):
    N = 300
    assert count_L_upto(k, N) == sum(1 for L in range(1, N + 1) if repunit(L) % k == 0)    # (3)
print("chữ số d bất kỳ, bội 0/1, đếm các L thoả: khớp vét cạn")

chữ số d bất kỳ, bội 0/1, đếm các L thoả: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Số lớn | định nghĩa | `O(k²)` | `O(k)` | oracle `k ≲ 600` |
| 2. Lặp số dư `k` bước | `r = (10r+1) mod k` | `O(k)` | `O(1)` | đơn giản |
| 3. Loại sớm + lặp | `k` chẵn/chia 5 ⇒ `-1` | `O(k)` | `O(1)` | **đáp án chính** |
| 4. Tập số dư | chu trình | `O(k)` | `O(k)` | không cần cận `k` |
| 5. Bậc của `10` mod `9k` | số học | `O(√k)` | `O(1)` | nhanh nhất |

Mọi cách khớp oracle số lớn với `k ≤ 600`, khớp nhau với `k ≤ 3000`, và Cách 3 khớp Cách 5 trên mẫu tới `10^5`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [32]:
print([Solution().smallestRepunitDivByK(k) for k in (1, 2, 3, 7, 11, 17, 5, 9, 13, 100000)])

[1, -1, 3, 6, 2, 16, -1, 9, 6, -1]


### Bài 2 (tính tay)

`k = 13`. (a) Tính các số dư `r_L = (10 r_{L−1} + 1) mod 13` cho `L = 1, 2, …` cho tới khi gặp `0`. (b) Đáp án? (c) Kiểm tra `111111` có chia hết cho `13` không? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cách lặp số dư:

In [33]:
def remainder_todo(k):
    if k % 2 == 0 or k % 5 == 0:
        return -1
    r = 0
    for L in range(1, k + 1):
        pass            # TODO 1: r = (r * 10 + 1) % k
        pass            # TODO 2: nếu r == 0 thì trả L
    return -1


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(remainder_todo(k) == w for k, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **bậc của `10` modulo `9k`**:

In [34]:
class MySolution:
    def smallestRepunitDivByK(self, k):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().smallestRepunitDivByK(k) == w for k, w in examples)

### Bài 5 (mini challenge): chữ số `d` bất kỳ

Viết `smallest_len(k, d)` trả về độ dài nhỏ nhất `L` sao cho số gồm `L` chữ số `d` chia hết cho `k` (`d ∈ 1..9`), hoặc `-1`. Kiểm bằng số lớn (`brute_digit_d` ở mục 15). Gợi ý: cận lặp vẫn là `k` bước.

In [35]:
def smallest_len(k, d):
    pass            # TODO


# Khi làm xong:
# assert smallest_len(7, 3) == 6 and smallest_len(9, 3) == 3 and smallest_len(4, 2) == -1 and smallest_len(4, 4) == 1

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `k` chẵn hoặc bội của `5` ⇒ `-1`.
- Mức 2: `k = 9` ⇒ `9`; `k = 11` ⇒ `2` (`11 | 11`).
- Mức 3: `1 -1 3 6 2 16 -1 9 6 -1`.

**Bài 2**

- Mức 1: `r_1 = 1`, `r_2 = (10·1 + 1) mod 13 = 11`.
- Mức 2: `r_3 = (110 + 1) mod 13 = 111 mod 13 = 7` (`111 = 8·13 + 7`); `r_4 = 71 mod 13 = 6`; `r_5 = 61 mod 13 = 9`; `r_6 = 91 mod 13 = 0`.
- Mức 3: danh sách số dư `[1, 11, 7, 6, 9, 0]` ⇒ đáp án `6` (`111111 = 13 · 8547`).

**Bài 3**

- Mức 1: `r = (r * 10 + 1) % k`.
- Mức 2: `if r == 0: return L`.
- Mức 3: giống Cách 3.

**Bài 4**

- Mức 1: loại `gcd(k, 10) != 1`.
- Mức 2: `m = 9 * k`, `order = phi(m)`.
- Mức 3: với mỗi ước nguyên tố `p` của `order`, nếu `pow(10, order // p, m) == 1` thì `order //= p` (lặp).

**Bài 5**

- Mức 1: `r = (r * 10 + d) % k`.
- Mức 2: lặp `L = 1..k`.
- Mức 3: trả `L` khi `r == 0`, hết vòng trả `-1`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [36]:
vals = [Solution().smallestRepunitDivByK(k) for k in (1, 2, 3, 7, 11, 17, 5, 9, 13, 100000)]
print(vals)
assert vals == [1, -1, 3, 6, 2, 16, -1, 9, 6, -1]
print("Bài 1 OK")

[1, -1, 3, 6, 2, 16, -1, 9, 6, -1]
Bài 1 OK


### Bài 2

In [37]:
rs = remainder_trace(13, 30)
print(rs)
assert rs == [1, 11, 7, 6, 9, 0] and len(rs) == 6
assert Solution().smallestRepunitDivByK(13) == 6 and 111111 % 13 == 0 and 111111 // 13 == 8547
print("Bài 2 OK: 6 bước, 111111 = 13 · 8547")

[1, 11, 7, 6, 9, 0]
Bài 2 OK: 6 bước, 111111 = 13 · 8547


### Bài 3

In [38]:
def remainder_answer(k):
    if k % 2 == 0 or k % 5 == 0:
        return -1
    r = 0
    for L in range(1, k + 1):
        r = (r * 10 + 1) % k
        if r == 0:
            return L
    return -1


assert all(remainder_answer(k) == w for k, w in examples)
for k in range(1, 400):
    assert remainder_answer(k) == smallest_bigint(k)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [39]:
class MySolutionAnswer:
    def smallestRepunitDivByK(self, k):
        if math.gcd(k, 10) != 1:
            return -1
        m = 9 * k
        order = euler_phi(m)
        for p in factorize(order):
            while order % p == 0 and pow(10, order // p, m) == 1:
                order //= p
        return order


assert all(MySolutionAnswer().smallestRepunitDivByK(k) == w for k, w in examples)
for k in range(1, 800):
    assert MySolutionAnswer().smallestRepunitDivByK(k) == smallest_early_exit(k)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [40]:
def smallest_len(k, d):
    r = 0
    for L in range(1, k + 1):
        r = (r * 10 + d) % k
        if r == 0:
            return L
    return -1


assert smallest_len(7, 3) == 6 and smallest_len(9, 3) == 3 and smallest_len(4, 2) == -1 and smallest_len(4, 4) == 1
for k in range(1, 150):
    for d in range(1, 10):
        assert smallest_len(k, d) == brute_digit_d(k, d)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (số dạng đặc biệt chia hết cho `k` → mô phỏng số dư, cận `k` bước):**

```python
class Solution:
    def smallestRepunitDivByK(self, k: int) -> int:
        if k % 2 == 0 or k % 5 == 0:
            return -1
        r = 0
        for L in range(1, k + 1):
            r = (r * 10 + 1) % k
            if r == 0:
                return L
        return -1
```

**Checklist trước khi submit:**

- [ ] Không dựng số lớn; chỉ giữ số dư `(r * 10 + 1) % k` (có ngoặc)?
- [ ] Lặp đủ `k` bước (`range(1, k + 1)`), đáp án có thể bằng `k`?
- [ ] Loại sớm `k` chẵn hoặc chia hết `5` ⇒ `-1` (hoặc dùng `visited`/cận `k`)?
- [ ] Trả độ dài `L` (không phải `L − 1`); `k = 1` ⇒ `1`?
- [ ] Đã thử `k = 1, 2, 3, 7, 11, 17, 100000`?
- [ ] Hiểu vì sao `gcd(k, 10) = 1` luôn có đáp án (định lý Euler)?
- [ ] Đúng tên `class Solution`, `smallestRepunitDivByK(self, k)`, không in debug.